In [245]:
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
import os
from langgraph.graph import StateGraph, END, MessagesState
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing_extensions import TypedDict, Literal, Annotated
from langchain.messages import HumanMessage, SystemMessage, AnyMessage
from langgraph.graph.message import add_messages
from langchain.tools import tool
from tavily import TavilyClient
from pydantic import BaseModel, Field
from pprint import pprint
from IPython.display import Markdown, display, Image
from langgraph.prebuilt import ToolNode

In [246]:
load_dotenv(".env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [247]:
basic_llm = ChatGroq(model="llama-3.1-8b-instant", api_key = GROQ_API_KEY, temperature=0)
advanced_llm = ChatGroq(model="llama-3.1-8b-instant", api_key = GROQ_API_KEY, temperature=0)

In [248]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

In [249]:
# Prompts - Read Content from Files
f = open("./prompts/fetching.md")
FETCHING_PROMPT = f.read()
f = open("./prompts/generations.md")
GENERATION_PROMPT = f.read()
f = open("./prompts/optimisation.md")
OPTIMISATION_PROMPT = f.read()
f = open("./prompts/scoring.md")
SCORING_PROMPT = f.read()
f = open("./prompts/validation.md")
VALIDATION_PROMPT = f.read()

In [250]:
@tool
def getDataFromExcelFile(customerName: str):
    """
        Retrieves customer records from the Excel file.
        Args:
            customerName: The name of the customer to look up in execel file.
        Return:
            return the customer data or no Data available here.
    """
    # Logic When data is Ready (pandas)
    return "No Data Available Here."

fetcher_agent = create_agent(
    model=basic_llm,
    tools=[getDataFromExcelFile],
    system_prompt=FETCHING_PROMPT
)

client_data = """Aya spent 10 dollar in t-shirt"""

class CustomerData(TypedDict):
    customer_data: str

query = f"USER_PROMPT = {client_data}"
clean_customer_data = ""
try:
    customer_data = fetcher_agent.invoke(input={"messages": [{"role": "user", "content": query}]})
    clean_customer_data = customer_data.get("messages")[-1].content
    print(clean_customer_data)
except:
    clean_customer_data = "no data available"
    print("ERROR: ", clean_customer_data)

No data available.


In [251]:
# Generation Agent
@tool
def getHowToWriteMarketingOffre(query: str, max_results=3):
    """
        Searches the internet for proven marketing copy techniques and offer structures.

        Args:
            query: Search query for marketing offer strategies.
    """
    response = tavily_client.search(query=query, max_results=max_results)
    results = response.get("results", [])
    content = []
    for res in results:
        content.append({"url": res.get("url", ""), "content": res.get("content", "")})
    return content


def check_policy_rules():
    """
    Validates offer compliance against company promotional guidelines.
    """
    return "the discount should be between 10% and 50%"
policies = check_policy_rules()

offer_agent = create_agent(
    model=basic_llm,
    tools=[getHowToWriteMarketingOffre],
    system_prompt=GENERATION_PROMPT
)

query = f"""
    USER_PROMPT = {client_data} \n\n
    Customer Data = {clean_customer_data} \n\n
    Policies = {policies}
"""
offre = offer_agent.invoke(input={"messages": [{"role": "user", "content": query}]})
offre = Markdown(offre.get("messages")[-1].content)
offre

Based on the customer data and policies provided, I will create a generic welcome promotion since the customer data states "No Data Available".

**Welcome Promotion**

Dear valued customer,

We're excited to welcome you to our t-shirt store! As a new customer, we'd like to offer you a special discount on your first purchase.

**Discount Offer**

Get 20% off your first purchase of any t-shirt. Simply use the code WELCOME20 at checkout to redeem your discount.

**Why Choose Us?**

At our store, we offer a wide range of high-quality t-shirts that are perfect for any occasion. Our designs are unique and stylish, and we're confident you'll find something that suits your taste.

**How to Redeem**

To redeem your discount, simply follow these steps:

1. Browse our collection of t-shirts and select the one you'd like to purchase.
2. Add the t-shirt to your cart and proceed to checkout.
3. Enter the code WELCOME20 at checkout to apply your discount.
4. Complete your purchase and enjoy your new t-shirt!

**Terms and Conditions**

* The discount offer is valid for new customers only.
* The discount code WELCOME20 can only be used once per customer.
* The discount offer is valid for a limited time only, so be sure to redeem it before it expires.

Thank you for choosing our store, and we hope you enjoy your new t-shirt!

Best regards,
[Your Name]

**README**

* Discount offer: 20% off first purchase
* Discount code: WELCOME20
* Valid for new customers only
* Limited time offer

In [252]:
# Scoring Agent
scoring_agent = create_agent(
    model=basic_llm,
    tools=[],
    system_prompt=SCORING_PROMPT
)

query = f"""
    - Offer to Score:\n\n
        {offre} \n\n
    - Policies = {policies}
"""

score = scoring_agent.invoke(input={"messages": [{"role": "user", "content": query}]})
clean_score = Markdown(score.get("messages")[-1].content)
clean_score

**Score**: 50 / 100
**Policy Compliance**: PASSED
**Areas for Improvement**:
- Discount is at the lower end of the allowed range.

In [253]:
# Optimisation Agent
optimisation_agent = create_agent(
    model=basic_llm,
    tools=[],
    system_prompt=OPTIMISATION_PROMPT
)

query = f"""
    - Optimise this Offre:\n\n
        {offre}
    - Offre Score: \n\n
        {clean_score}
    - Policies: \n\n
        {policies}
"""

new_offre = optimisation_agent.invoke(input={"messages": [{"role": "user", "content": query}]})
clean_new_offre = Markdown(new_offre.get("messages")[-1].content)
clean_new_offre

**Improved Offer**

Get 20% Off Your First Purchase

We're excited to welcome you to our community! As a valued customer, we're offering you an exclusive 20% discount on your first purchase. Use code WELCOME20 at checkout to redeem your discount.

This offer is valid for a limited time only, so don't miss out. Browse our collection of high-quality products and experience the best value for your money.

**Terms and Conditions:**

- The 20% discount is applicable on the first purchase only.
- The discount code WELCOME20 can be used once per customer.
- The offer is valid for a limited time only and can be withdrawn at any time.
- The discount is not applicable on sale items, gift cards, or other promotional offers.

**Pricing:**

- Regular price: $100
- Discounted price: $80 (20% off)

**How to Redeem:**

1. Browse our collection and add your desired products to the cart.
2. Enter the code WELCOME20 at checkout.
3. Click "Apply" to receive your 20% discount.
4. Complete your purchase to enjoy your savings.

Don't miss out on this amazing opportunity to save 20% on your first purchase. Use code WELCOME20 now and experience the best value for your money!